[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YamKhoonTor/datums-book/blob/main/notebooks/Chapter_15_Grid_to_ground.ipynb)

Companion notebook to *Datums, Coordinate Systems and Map Projections* by Yam Khoon Tor. Choose *Runtime → Run all*, compare with the output stored beneath each listing, then change the values in the *Try it* cell and run it again.

# Chapter 15. Grid to ground

Companion notebook to *Datums, Coordinate Systems and Map Projections: A Practical Guide for Surveyors and Engineers, with SVY21 Worked Examples*, by Yam Khoon Tor.

*Copyright © 2026 Yam Khoon Tor. All rights reserved. Readers who have bought the book may run, copy and modify this notebook for personal study and their own professional work; it may not be republished, distributed or sold without the author's written permission. Provided as a teaching aid, without warranty: verify results independently before use in real work.*

**How to use this notebook.** Run the cells in order from the top: select a cell and press the run button (or Shift+Enter), or choose *Run all* from the menu. Each listing is exactly as printed in the book. Beneath it the notebook stores the output of the run, and repeats the output printed in the book so that the two can be compared. If an unaltered notebook gives different numbers, something is wrong with the installation and should be put right before going on. The last section, **Try it**, repeats an example with the input values gathered at the top of the cell, to be changed freely.

Requirements: Python 3.9 or later; no other library is needed.

## Listing 15.1  Reduction between ground and grid distances

In [1]:
import math

R_SG = 6356777.0            # mean radius of curvature in Singapore (m)
E0_SVY21 = 28001.642        # false easting of the SVY21 grid (m)


def horizontal_distance(slope, h1, h2):
    """Horizontal distance at the mean height of the line, from the slope
    distance and the heights of its two ends."""
    return math.sqrt(slope ** 2 - (h2 - h1) ** 2)


def elevation_factor(h_ellipsoidal, R=R_SG):
    """Factor reducing a horizontal distance at ellipsoidal height h
    to the ellipsoid."""
    return R / (R + h_ellipsoidal)


def point_scale_factor(E, E0=E0_SVY21, k0=1.0, R=R_SG):
    """Point scale factor of a Transverse Mercator grid at easting E."""
    x = E - E0
    return k0 * (1.0 + x * x / (2.0 * k0 * k0 * R * R))


def line_scale_factor(E1, E2, E0=E0_SVY21, k0=1.0, R=R_SG):
    """Line scale factor between two eastings."""
    x1, x2 = E1 - E0, E2 - E0
    return k0 * (1.0 + (x1 * x1 + x1 * x2 + x2 * x2) / (6.0 * k0 * k0 * R * R))


def ground_to_grid(horizontal, h_mean, E1, E2, **grid):
    """Grid distance from a horizontal ground distance.
    Returns the distance, the elevation factor and the line scale factor."""
    ef = elevation_factor(h_mean)
    K = line_scale_factor(E1, E2, **grid)
    return horizontal * ef * K, ef, K


def grid_to_ground(grid_distance, h_mean, E1, E2, **grid):
    """Horizontal ground distance from a grid distance.
    Returns the distance, the elevation factor and the line scale factor."""
    ef = elevation_factor(h_mean)
    K = line_scale_factor(E1, E2, **grid)
    return grid_distance / (ef * K), ef, K


def ppm(factor):
    """Departure of a scale factor from unity, in parts per million."""
    return (factor - 1.0) * 1.0e6


if __name__ == "__main__":
    # Example 15.1: a measured slope distance reduced to the SVY21 grid
    slope = 1254.386
    h1 = 12.400 + 1.550 + 9.7     # reduced level + instrument height + undulation
    h2 = 51.814 + 1.400 + 9.7     # reduced level + target height + undulation
    D = horizontal_distance(slope, h1, h2)
    grid, ef, K = ground_to_grid(D, (h1 + h2) / 2.0, 8500.0, 9712.9)
    print(f"horizontal = {D:.4f} m")
    print(f"EF = {ppm(ef):+.2f} ppm   K = {ppm(K):+.2f} ppm   "
          f"combined = {ppm(ef * K):+.2f} ppm")
    print(f"grid distance = {grid:.4f} m")

    # Example 15.2: a grid distance converted for setting out
    ground, ef, K = grid_to_ground(350.000, 14.0 + 9.7, 4200.0, 4520.0)
    print(f"EF = {ppm(ef):+.2f} ppm   K = {ppm(K):+.2f} ppm   "
          f"ground distance = {ground:.4f} m")

    # Example 15.3: the line of Example 15.1 on UTM zone 48 North
    grid, ef, K = ground_to_grid(D, (h1 + h2) / 2.0, 350704.461, 351917.369,
                                 E0=500000.0, k0=0.9996)
    print(f"UTM: K = {ppm(K):+.1f} ppm   grid distance = {grid:.4f} m")

horizontal = 1253.7713 m
EF = -6.81 ppm   K = +4.42 ppm   combined = -2.39 ppm
grid distance = 1253.7683 m
EF = -3.73 ppm   K = +6.92 ppm   ground distance = 349.9989 m
UTM: K = -126.3 ppm   grid distance = 1253.6044 m


**Expected output, as printed in the book:**

```
horizontal = 1253.7713 m
EF = -6.81 ppm   K = +4.42 ppm   combined = -2.39 ppm
grid distance = 1253.7683 m
EF = -3.73 ppm   K = +6.92 ppm   ground distance = 349.9989 m
UTM: K = -126.3 ppm   grid distance = 1253.6044 m
```

## Try it

Change the values between the two dashed lines and run the cell again. Nothing here alters the listings above.

In [2]:
# ---- Change these values, then run the cell ----------------------
slope = 1254.386                     # measured slope distance (m)
RL1, hi = 12.400, 1.550              # reduced level of station, instrument height
RL2, ht = 51.814, 1.400              # reduced level of target, target height
undulation_N = 9.7                   # geoid undulation in Singapore (m)
E1, E2 = 8500.0, 9712.9              # SVY21 eastings of the two ends (m)
# -------------------------------------------------------------------

h1 = RL1 + hi + undulation_N         # ellipsoidal height of the instrument
h2 = RL2 + ht + undulation_N         # ellipsoidal height of the target
D = horizontal_distance(slope, h1, h2)
grid, ef, K = ground_to_grid(D, (h1 + h2) / 2.0, E1, E2)
print(f"horizontal distance = {D:.4f} m")
print(f"elevation factor  {ppm(ef):+.2f} ppm")
print(f"line scale factor {ppm(K):+.2f} ppm")
print(f"combined          {ppm(ef * K):+.2f} ppm")
print(f"grid distance = {grid:.4f} m   (grid minus ground = {(grid - D) * 1000:+.1f} mm)")

horizontal distance = 1253.7713 m
elevation factor  -6.81 ppm
line scale factor +4.42 ppm
combined          -2.39 ppm
grid distance = 1253.7683 m   (grid minus ground = -3.0 mm)
